# Migrate All Elasticsearch Indices to FAISS

This notebook migrates all local Elasticsearch indices to memory-efficient FAISS indexes.

## Benefits of FAISS over Elasticsearch:
- **Lower RAM usage**: IVF_PQ uses ~48 bytes/vector vs ~6KB for flat storage
- **No server overhead**: Local files, no ES daemon needed
- **Faster cold start**: Load via memory-mapping
- **Better for development**: Easier to version control and share

## Prerequisites:
```bash
# Ensure local Elasticsearch is running
cd elastic-start-local && ./start.sh

# Install dependencies
pip install faiss-cpu pandas pyarrow elasticsearch python-dotenv tqdm
```

## Migration Modes:
1. **Use existing vectors** (faster): Copies vectors from ES, no re-embedding
2. **Re-embed** (slower): Fetches text and creates new embeddings

## Output:
Each index is saved to `data/faiss/<index_name>/`:
```
data/faiss/wiki_full_l/
├── faiss/
│   ├── index.faiss       # FAISS binary index
│   └── docstore.sqlite   # Documents + metadata (on-disk)
└── migration_state.json  # For resuming
```

In [1]:
import os
import sys
import logging
from pathlib import Path
from typing import List, Dict

import dotenv
import pandas as pd
from elasticsearch import Elasticsearch
from tqdm.notebook import tqdm

from config import DATA_DIR

# Load environment
dotenv.load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress noisy HTTP logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("elastic_transport").setLevel(logging.WARNING)

print("✓ Imports loaded")

/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Imports loaded


## 1. Connect to Elasticsearch and Discover Indices

In [2]:
# Elasticsearch connection
ES_URL = "http://localhost:9200"
ES_USER = os.getenv("ELASTICSEARCH_USERNAME")
ES_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD")

# Connect
if ES_USER and ES_PASSWORD:
    es_client = Elasticsearch(ES_URL, basic_auth=(ES_USER, ES_PASSWORD), request_timeout=300)
else:
    es_client = Elasticsearch(ES_URL, request_timeout=300)

if not es_client.ping():
    raise ConnectionError(f"Could not connect to Elasticsearch at {ES_URL}")

print(f"✓ Connected to Elasticsearch at {ES_URL}")

✓ Connected to Elasticsearch at http://localhost:9200


In [3]:
# Discover all indices (exclude system indices starting with '.')
indices_raw = es_client.cat.indices(format='json')
indices = [
    idx for idx in indices_raw 
    if not idx['index'].startswith('.')
]

# Display as DataFrame
df_indices = pd.DataFrame([
    {
        'index': idx['index'],
        'docs': int(idx['docs.count']) if idx['docs.count'] != 'null' else 0,
        'size': idx['store.size'],
        'health': idx['health'],
    }
    for idx in indices
]).sort_values('docs', ascending=False)

print(f"\nFound {len(df_indices)} indices:\n")
display(df_indices)

AuthorizationException: AuthorizationException(403, 'security_exception', 'current license is non-compliant for [security]')

## 2. Configure Migration Settings

### Strategy Selection Guide:

| Strategy | RAM Usage | Speed | Accuracy | Best For |
|----------|-----------|-------|----------|----------|
| `vector` (flat) | High (6KB/vec) | Slow | 100% | Small datasets (< 100K docs) |
| `hnsw` | Medium (13KB/vec) | Fast | ~95% | Medium datasets (< 1M docs) |
| `ivfpq` | **Low (48 bytes/vec)** | Fast | ~90% | **Large datasets (> 1M docs)** |
| `opq_ivfpq` | Low (48 bytes/vec) | Fast | ~92% | Better accuracy at same size |
| `ivfpq_disk` | **Minimal (centroids only)** | Medium | ~90% | **Billion-scale** |

### Memory Estimate:
For **4M documents** with **1024-dim embeddings**:
- `vector`: ~23 GB RAM
- `hnsw`: ~51 GB RAM  
- `ivfpq`: **~200 MB RAM** ← Recommended
- `ivfpq_disk`: **~50 MB RAM**

In [ ]:
# ============================================================================
# MIGRATION CONFIGURATION
# ============================================================================

# Which indices to migrate (empty list = all indices)
INDICES_TO_MIGRATE = []  # e.g., ['wiki_full_l', 'wiki_small_s']

# FAISS strategy
STRATEGY = "ivfpq"  # Options: vector, hnsw, ivfpq, opq_ivfpq, ivfpq_disk

# Migration mode
USE_EXISTING_VECTORS = True  # True = copy ES vectors (fast), False = re-embed (slow)

# Output directory
OUTPUT_BASE_DIR = DATA_DIR / "faiss"

# Memory settings
LOW_RAM = False  # Set True if you have < 8GB RAM available
MAX_RAM_MB = 4096
BATCH_SIZE = 1000 if not LOW_RAM else 500

# Processing settings
CHECKPOINT_EVERY = 10_000  # Save progress every N docs
DEDUPLICATE = True  # Remove duplicate documents
RESUME = True  # Resume interrupted migrations

# IVF_PQ parameters (for ivfpq strategy)
# These are tuned for ~4M docs with 1024-dim embeddings
IVFPQ_NLIST = 4096  # Number of Voronoi cells (~sqrt(n_docs))
IVFPQ_M = 32  # Sub-quantizers (must divide embedding dim)
IVFPQ_NBITS = 8  # Bits per sub-quantizer (8 = 256 centroids)
IVFPQ_NPROBE = 64  # Cells to search (higher = better recall, slower)

# Embedding settings (only if USE_EXISTING_VECTORS = False)
EMBEDDING_PROVIDER = "modal"  # Options: modal, openai, huggingface
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
GPU_BATCH_SIZE = 64         # Forward-pass batch size on GPU (modal)
REQUEST_BATCH_SIZE = 100    # Texts per Modal request batch
NORMALISE_EMBEDDINGS = True # Normalise embedding vectors

# Testing limits (set to None for full migration)
MAX_DOCS_PER_INDEX = None  # Limit docs per index for testing

print("✓ Configuration set:")
print(f"  Strategy: {STRATEGY}")
print(f"  Mode: {'Use existing vectors' if USE_EXISTING_VECTORS else 'Re-embed'}")
print(f"  Output: {OUTPUT_BASE_DIR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Low RAM mode: {LOW_RAM}")

## 3. Estimate Memory Requirements

Before migrating, let's check if we have enough RAM for the selected strategy.

In [ ]:
from rag.faiss_rag_service import FaissRagService, MemoryConfig
from rag.utils import IndexingConfig

# Create a temporary service instance for estimates
temp_config = IndexingConfig(
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    gpu_batch_size=GPU_BATCH_SIZE,
    request_batch_size=REQUEST_BATCH_SIZE,
    normalise_embeddings=NORMALISE_EMBEDDINGS,
)

temp_service = FaissRagService(
    config=temp_config,
    strategy=STRATEGY,
    memory_config=MemoryConfig(max_ram_mb=MAX_RAM_MB),
    ivfpq_m=IVFPQ_M,
    ivfpq_nlist=IVFPQ_NLIST,
)

# Estimate for each index
print("Memory estimates for each index:\n")
print(f"{'Index':<30} {'Docs':>12} {'Selected Strategy':>20} {'All Strategies (MB)':<50}")
print("=" * 120)

for _, row in df_indices.iterrows():
    if row['docs'] == 0:
        continue
    
    # Assume 1024-dim embeddings (adjust if different)
    estimates = temp_service.estimate_index_memory(row['docs'], dim=1024)
    selected_mb = estimates.get(STRATEGY, 0)
    
    # Color-code based on budget
    status = "✓" if selected_mb < MAX_RAM_MB * 0.8 else "⚠️"
    
    all_estimates = ", ".join([f"{k}:{v:.0f}" for k, v in estimates.items()])
    
    print(f"{status} {row['index']:<28} {row['docs']:>12,} {selected_mb:>18,.0f} MB   {all_estimates}")

print("\n" + "=" * 120)
print(f"RAM budget: {MAX_RAM_MB:,} MB")
print("\nLegend: ✓ = Fits in budget, ⚠️ = May exceed RAM budget")

## 4. Run Migration

This will migrate all selected indices using the migration script.

In [ ]:
import subprocess
import time
from datetime import datetime

# Determine which indices to migrate
if INDICES_TO_MIGRATE:
    indices_to_process = [idx for idx in df_indices['index'].tolist() if idx in INDICES_TO_MIGRATE]
else:
    indices_to_process = df_indices['index'].tolist()

print(f"\nMigrating {len(indices_to_process)} indices...\n")

# Track results
results = []
start_time = time.time()

for idx_name in indices_to_process:
    print(f"\n{'='*80}")
    print(f"Migrating: {idx_name}")
    print(f"{'='*80}\n")
    
    # Build command
    output_dir = OUTPUT_BASE_DIR / idx_name
    
    cmd = [
        "python", "scripts/migrate_es_to_faiss.py",
        "--es-index", idx_name,
        "--output-dir", str(output_dir),
        "--strategy", STRATEGY,
        "--batch-size", str(BATCH_SIZE),
        "--max-ram-mb", str(MAX_RAM_MB),
        "--ivfpq-nlist", str(IVFPQ_NLIST),
        "--ivfpq-m", str(IVFPQ_M),
        "--ivfpq-nbits", str(IVFPQ_NBITS),
        "--ivfpq-nprobe", str(IVFPQ_NPROBE),
    ]
    
    if USE_EXISTING_VECTORS:
        cmd.append("--use-existing-vectors")
    else:
        cmd.extend(["--embedding-provider", EMBEDDING_PROVIDER])
        cmd.extend(["--embedding-model", EMBEDDING_MODEL])
        cmd.extend(["--gpu-batch-size", str(GPU_BATCH_SIZE)])
        cmd.extend(["--request-batch-size", str(REQUEST_BATCH_SIZE)])
        cmd.extend(["--normalise-embeddings", str(NORMALISE_EMBEDDINGS).lower()])
    
    if LOW_RAM:
        cmd.append("--low-ram")
    
    if RESUME:
        cmd.append("--resume")
    
    if not DEDUPLICATE:
        cmd.append("--no-dedupe")
    
    if MAX_DOCS_PER_INDEX:
        cmd.extend(["--max-docs", str(MAX_DOCS_PER_INDEX)])
    
    # Run migration
    migration_start = time.time()
    try:
        result = subprocess.run(
            cmd,
            capture_output=False,  # Stream output to notebook
            text=True,
        )
        
        migration_time = time.time() - migration_start
        success = result.returncode == 0
        
        results.append({
            'index': idx_name,
            'success': success,
            'time_seconds': migration_time,
            'output_dir': str(output_dir),
        })
        
        if success:
            print(f"\n✓ {idx_name} migrated successfully in {migration_time/60:.1f} minutes")
        else:
            print(f"\n✗ {idx_name} migration failed (exit code: {result.returncode})")
    
    except Exception as e:
        print(f"\n✗ {idx_name} migration error: {e}")
        results.append({
            'index': idx_name,
            'success': False,
            'time_seconds': 0,
            'output_dir': str(output_dir),
            'error': str(e),
        })

total_time = time.time() - start_time

print(f"\n\n{'='*80}")
print(f"All migrations complete in {total_time/60:.1f} minutes")
print(f"{'='*80}\n")

## 5. Migration Summary

In [ ]:
# Display results
df_results = pd.DataFrame(results)

if len(df_results) > 0:
    df_results['time_minutes'] = df_results['time_seconds'] / 60
    df_results['status'] = df_results['success'].apply(lambda x: '✓ Success' if x else '✗ Failed')
    
    print("\nMigration Summary:\n")
    display(df_results[['index', 'status', 'time_minutes', 'output_dir']])
    
    # Statistics
    n_success = df_results['success'].sum()
    n_failed = len(df_results) - n_success
    total_time_min = df_results['time_seconds'].sum() / 60
    
    print(f"\nStatistics:")
    print(f"  Successful: {n_success} / {len(df_results)}")
    print(f"  Failed: {n_failed}")
    print(f"  Total time: {total_time_min:.1f} minutes")
    print(f"  Average time per index: {total_time_min/len(df_results):.1f} minutes")
else:
    print("No indices migrated.")

## 6. Verify Migrated Indices

Let's verify the FAISS indices were created correctly.

In [ ]:
import json

verification = []

for _, row in df_results.iterrows():
    if not row['success']:
        continue
    
    output_dir = Path(row['output_dir'])
    faiss_dir = output_dir / "faiss"
    
    # Check files exist
    index_file = faiss_dir / "index.faiss"
    docstore_file = faiss_dir / "docstore.sqlite"
    state_file = output_dir / "migration_state.json"
    
    files_exist = index_file.exists() and docstore_file.exists()
    
    # Load state
    docs_migrated = 0
    completed = False
    if state_file.exists():
        with open(state_file) as f:
            state = json.load(f)
            docs_migrated = state.get('docs_processed', 0)
            completed = state.get('completed', False)
    
    # Get file sizes
    index_size_mb = index_file.stat().st_size / (1024**2) if index_file.exists() else 0
    docstore_size_mb = docstore_file.stat().st_size / (1024**2) if docstore_file.exists() else 0
    
    verification.append({
        'index': row['index'],
        'files_exist': '✓' if files_exist else '✗',
        'completed': '✓' if completed else '✗',
        'docs_migrated': docs_migrated,
        'index_size_mb': index_size_mb,
        'docstore_size_mb': docstore_size_mb,
        'total_size_mb': index_size_mb + docstore_size_mb,
    })

df_verify = pd.DataFrame(verification)

if len(df_verify) > 0:
    print("\nVerification Results:\n")
    display(df_verify)
    
    total_size = df_verify['total_size_mb'].sum()
    print(f"\nTotal FAISS storage: {total_size:,.1f} MB")
else:
    print("No successful migrations to verify.")

## 7. Test Migrated Index

Let's test loading and searching one of the migrated indices.

In [ ]:
# Pick the first successful migration to test
test_results = df_results[df_results['success'] == True]

if len(test_results) > 0:
    test_index = test_results.iloc[0]['index']
    test_dir = Path(test_results.iloc[0]['output_dir'])
    
    print(f"Testing index: {test_index}")
    print(f"Loading from: {test_dir}\n")
    
    # Create service and load index
    test_config = IndexingConfig(
        embedding_provider=EMBEDDING_PROVIDER,
        embedding_model=EMBEDDING_MODEL,
        gpu_batch_size=GPU_BATCH_SIZE,
        request_batch_size=REQUEST_BATCH_SIZE,
        normalise_embeddings=NORMALISE_EMBEDDINGS,
    )
    
    service = FaissRagService(
        config=test_config,
        strategy=STRATEGY,
        memory_config=MemoryConfig(
            use_mmap=True,  # Use memory-mapping for low RAM usage
            max_ram_mb=MAX_RAM_MB,
        ),
        ivfpq_nlist=IVFPQ_NLIST,
        ivfpq_m=IVFPQ_M,
        ivfpq_nbits=IVFPQ_NBITS,
        ivfpq_nprobe=IVFPQ_NPROBE,
    )
    
    # Load with memory-mapping
    service.load_faiss_store(str(test_dir), use_mmap=True)
    
    # Get stats
    stats = service.get_index_stats()
    print("\nIndex Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")
    
    # Test search (if embeddings available)
    if not USE_EXISTING_VECTORS:
        print("\n\nTest Search:")
        test_query = "What is machine learning?"
        print(f"Query: {test_query}\n")
        
        results = service.retrieve_documents(test_query, top_k=3)
        
        for i, doc in enumerate(results, 1):
            print(f"\n{i}. {doc.page_content[:200]}...")
            print(f"   Metadata: {doc.metadata}")
    else:
        print("\nNote: Search requires re-initializing with embeddings enabled.")
    
    print("\n✓ Index loaded and verified successfully!")
else:
    print("No successful migrations to test.")

## 8. Cleanup (Optional)

After verifying migrations, you can optionally delete the Elasticsearch indices to save disk space.

⚠️ **WARNING**: This is irreversible! Only run if you've verified the FAISS indices work correctly.

In [ ]:
# Set to True to enable cleanup
ENABLE_CLEANUP = False

if ENABLE_CLEANUP:
    print("⚠️  WARNING: This will DELETE Elasticsearch indices!\n")
    
    # Only delete successfully migrated indices
    successful_indices = df_results[df_results['success'] == True]['index'].tolist()
    
    print(f"Indices to delete: {successful_indices}\n")
    
    # Uncomment to actually delete
    # for idx_name in successful_indices:
    #     try:
    #         es_client.indices.delete(index=idx_name)
    #         print(f"✓ Deleted: {idx_name}")
    #     except Exception as e:
    #         print(f"✗ Failed to delete {idx_name}: {e}")
    
    print("\nTo actually delete indices, uncomment the deletion code above.")
else:
    print("Cleanup disabled (ENABLE_CLEANUP = False)")
    print("Set ENABLE_CLEANUP = True and run this cell to delete ES indices.")

## Summary

### What We Did:
1. ✓ Connected to local Elasticsearch
2. ✓ Discovered all indices
3. ✓ Estimated memory requirements
4. ✓ Migrated indices to FAISS
5. ✓ Verified migrations
6. ✓ Tested loading and search

### Next Steps:
- Use the FAISS indices in your RAG pipeline via `FaissRagService.load_faiss_store()`
- Optionally delete ES indices to save disk space (see cleanup section)
- Share FAISS indices by copying the output directory

### Memory Comparison:
With IVF_PQ strategy:
- **Elasticsearch**: ~23 GB for 4M docs (full vectors in RAM)
- **FAISS (IVF_PQ)**: ~200 MB (compressed index)
- **FAISS (mmap)**: ~50 MB active RAM (OS loads pages on-demand)

**115x less RAM** with memory-mapped IVF_PQ! 🎉